In [ ]:
!nvidia-smi

In [ ]:
# ============================================
# PHASE 2 — REFERENCE STATE
# COMPLETE MODEL INPUT VERSION
# ============================================

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "application_train.csv"
MODEL_DIR = PROJECT_ROOT / "models"
preprocessor = joblib.load(MODEL_DIR / "baseline_preprocessor.joblib")
source_columns = set(pd.read_csv(DATA_PATH, nrows=0).columns)
reference_columns = [
    str(column)
    for column in preprocessor.feature_names_in_
    if str(column) in source_columns
]

X_reference = pd.read_csv(DATA_PATH, usecols=reference_columns)

# Fix known sentinel
X_reference["DAYS_EMPLOYED"] = X_reference[
    "DAYS_EMPLOYED"
].replace(365243, np.nan)

print("Reference data loaded:", X_reference.shape)

In [ ]:
#Apply the SAME feature engineering

def engineer_features(df):
    data = df.copy()

    data["AGE_YEARS"] = -data["DAYS_BIRTH"] / 365.25
    data["EMPLOYED_YEARS"] = -data["DAYS_EMPLOYED"] / 365.25

    def safe_divide(a, b):
        return a / b.replace(0, np.nan)

    data["CREDIT_TO_INCOME"] = safe_divide(
        data["AMT_CREDIT"], data["AMT_INCOME_TOTAL"]
    )

    data["ANNUITY_TO_INCOME"] = safe_divide(
        data["AMT_ANNUITY"], data["AMT_INCOME_TOTAL"]
    )

    data["CREDIT_TO_ANNUITY"] = safe_divide(
        data["AMT_CREDIT"], data["AMT_ANNUITY"]
    )

    data["GOODS_TO_CREDIT"] = safe_divide(
        data["AMT_GOODS_PRICE"], data["AMT_CREDIT"]
    )

    data["INCOME_PER_FAMILY_MEMBER"] = safe_divide(
        data["AMT_INCOME_TOTAL"], data["CNT_FAM_MEMBERS"]
    )

    data["INCOME_PER_CHILD"] = (
        data["AMT_INCOME_TOTAL"] /
        (data["CNT_CHILDREN"] + 1)
    )

    external_sources = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    data["EXT_SOURCE_MEAN"] = data[
        external_sources
    ].mean(axis=1)

    data["EXT_SOURCE_MAX"] = data[
        external_sources
    ].max(axis=1)

    return data


X_reference_fe = engineer_features(X_reference)
expected_columns = set(str(column) for column in preprocessor.feature_names_in_)
if set(X_reference_fe.columns) != expected_columns:
    missing = sorted(expected_columns - set(X_reference_fe.columns))
    unexpected = sorted(set(X_reference_fe.columns) - expected_columns)
    raise ValueError(f"Reference schema mismatch; missing={missing}, unexpected={unexpected}")
X_reference_fe = X_reference_fe.loc[:, preprocessor.feature_names_in_]

print("Reference after feature engineering:", X_reference_fe.shape)

In [ ]:
# Create reference distributions for every V1 preprocessor input.
reference_distribution = {}
quantiles = [0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1]
numeric_features = {
    str(name)
    for transformer_name, _, columns in preprocessor.transformers_
    if transformer_name == "num"
    for name in columns
}
categorical_features = {
    str(name)
    for transformer_name, _, columns in preprocessor.transformers_
    if transformer_name == "cat"
    for name in columns
}
if (
    numeric_features & categorical_features
    or numeric_features | categorical_features != expected_columns
):
    raise ValueError("Preprocessor numeric/categorical schema is inconsistent.")

for column in X_reference_fe.columns:
    if column in numeric_features:
        series = pd.to_numeric(X_reference_fe[column], errors="coerce")
        values = series.dropna()
        reference_distribution[column] = {
            "type": "numeric",
            "count": int(values.count()),
            "mean": float(values.mean()),
            "std": float(values.std()),
            "min": float(values.min()),
            "q01": float(values.quantile(0.01)),
            "q05": float(values.quantile(0.05)),
            "q25": float(values.quantile(0.25)),
            "q50": float(values.quantile(0.5)),
            "q75": float(values.quantile(0.75)),
            "q95": float(values.quantile(0.95)),
            "q99": float(values.quantile(0.99)),
            "max": float(values.max()),
            "missing_rate": float(series.isna().mean()),
            "quantiles": {
                str(q): float(values.quantile(q)) for q in quantiles
            },
        }
    else:
        series = X_reference_fe[column]
        categories = series.fillna("__MISSING__").astype(str)
        reference_distribution[column] = {
            "type": "categorical",
            "missing_rate": float(series.isna().mean()),
            "frequencies": {
                str(category): float(rate)
                for category, rate in categories.value_counts(
                    normalize=True
                ).sort_index().items()
            },
        }

print("Reference features:", len(reference_distribution))

In [ ]:
# Save the reference distribution.

MODEL_DIR.mkdir(exist_ok=True)

with open(
    MODEL_DIR / "reference_distribution.json",
    "w"
) as f:
    json.dump(
        reference_distribution,
        f,
        indent=4,
        allow_nan=False,
    )

print("Reference distribution saved.")